# Import Libraries

In [ ]:
# Analisi dati e Visualizzazione
import pandas as pd
import matplotlib.pyplot as plt
import time
import copy

# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
    ParameterGrid
)
from sklearn.pipeline import Pipeline

from pytorch_tabnet.pretraining import TabNetPretrainer
from pytorch_tabnet.tab_model import TabNetClassifier

# Metriche di Valutazione
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# Deep Learning (PyTorch) e Monitoraggio
import torch

# Our packages
from support_modules.utils import *
from support_modules.plot import *


from sklearn import set_config
set_config(transform_output="pandas")

# Global Variables

In [ ]:
SEED = 42
FILENAME = "../data/train.csv"

# Cerca la GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates rows and columns

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df.T.drop_duplicates(inplace=True).T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


## 2. Label extraction and train-test splitting

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

## 3. Pipeline

In [ ]:

def create_train_val_split(X, y, pipeline, test_size=0.25, random_state=42):
    X_train_raw, X_val_raw, Y_train, Y_val = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    X_train_tab = pipeline.structure_pipeline.fit_transform(X_train_raw)
    X_val_tab = pipeline.structure_pipeline.transform(X_val_raw)

    pipeline.cat_imputer.fit(X_train_tab)
    X_train_tab = pipeline.cat_imputer.transform(X_train_tab)
    X_val_tab = pipeline.cat_imputer.transform(X_val_tab)
    
    pipeline.num_imputer.fit(X_train_tab)
    X_train_tab = pipeline.num_imputer.transform(X_train_tab)
    X_val_tab = pipeline.num_imputer.transform(X_val_tab)
    
    pipeline.label_encoder.fit(X_train_tab)
    X_train_tab = pipeline.label_encoder.transform(X_train_tab)
    X_val_tab = pipeline.label_encoder.transform(X_val_tab)

    y_label_encoder = LabelEncoder()
    Y_train_encoded = y_label_encoder.fit_transform(Y_train)
    Y_val_encoded = y_label_encoder.transform(Y_val)
    
    # Convert to numpy
    X_train_np = pipeline.to_float.transform(X_train_tab)
    X_val_np = pipeline.to_float.transform(X_val_tab)
    Y_train_np = Y_train_encoded
    Y_val_np = Y_val_encoded 

    pipeline.is_fitted_ = True
    
    pipeline.save('tb_preprocessor.save')

    return {
        'X_train_np': X_train_np,
        'X_val_np': X_val_np,
        'Y_train_np': Y_train_np,
        'Y_val_np': Y_val_np,
        'cat_idxs': pipeline.get_cat_idxs(),
        'cat_dims': pipeline.get_cat_dims()
    }


In [ ]:
from support_modules.preprocessing import *

In [ ]:
from support_modules.constants import *

In [ ]:
def run_tabnet_grid_search(
    param_grid,
    X_train,
    Y_train,
    X_val,
    Y_val,
    cat_idxs,
    cat_dims,
    save_path="tabnet",
    load_best=False,
):
    best_loss = float("inf")
    best_balanced_acc = 0.0
    best_model = None
    best_params = None
    best_time = 0

    grid = list(ParameterGrid(param_grid))
    n_comb = len(grid)

    print("="*70)
    print(f"INIZIO GRID SEARCH TABNET")
    print(f"Totale combinazioni da testare: {n_comb}")
    print(f"Campioni di addestramento: {X_train.shape[0]}, Feature: {X_train.shape[1]}")
    print(f"Feature categoriche: {len(cat_idxs)}")
    print("="*70 + "\n")

    for i, params in enumerate(grid):
        print("\n" + "="*70)
        print(f"[ESECUZIONE {i+1}/{n_comb}] INIZIO")
        print(f"Configurazione: {params}")
        print("="*70)

        start_time = time.time()

        # Estrazione parametri con default
        n_d = params.get('n_d', 64)
        n_a = params.get('n_a', n_d)
        n_steps = params.get('n_steps', 5)
        gamma = params.get('gamma', 1.5)
        n_independent = params.get('n_independent', 2)
        n_shared = params.get('n_shared', 2)
        epsilon = params.get('epsilon', 1e-15)
        lr = params.get('lr', 0.02)
        momentum = params.get('momentum', 0.02)
        pretraining_ratio = params.get('pretraining_ratio', 0.5)
        batch_size = params.get('batch_size', 1024)
        patience = params.get('patience', 10)
        max_epochs = params.get('epochs', 100)

        # Pre-addestramento
        print(f"\n>>> Fase 1/2: Pre-addestramento non supervisionato (max {max_epochs} epoche, pazienza={patience})...")
        pretrain_start = time.time()
        
        unsupervised_model = TabNetPretrainer(
            n_d=n_d, n_a=n_d, n_steps=n_steps, gamma=gamma,
            n_independent=n_independent, n_shared=n_shared,
            cat_idxs=cat_idxs, cat_dims=cat_dims,
            lambda_sparse=1e-3,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=lr),
            mask_type="sparsemax",
            device_name=device.type,
            verbose=1
        )

        unsupervised_model.fit(
            X_train=X_train,
            eval_set=[X_val],
            max_epochs=max_epochs,
            patience=patience,
            batch_size=batch_size,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=False,
            pretraining_ratio=pretraining_ratio
        )
        
        pretrain_time = time.time() - pretrain_start
        print(f">>> Pre-addestramento completato in {pretrain_time:.2f}s ({pretrain_time/60:.2f}min)")

        # Addestramento
        print(f"\n>>> Fase 2/2: Addestramento supervisionato (max {max_epochs} epoche, pazienza={patience})...")
        train_start = time.time()
        
        model = TabNetClassifier(
            n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
            n_independent=n_independent, n_shared=n_shared,
            momentum=momentum,
            cat_idxs=cat_idxs, cat_dims=cat_dims,
            lambda_sparse=1e-3,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=lr),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params={"step_size":10, "gamma":0.9},
            mask_type="sparsemax",
            device_name=device.type,
            verbose=1
        )

        model.fit(
            X_train=X_train, y_train=Y_train,
            eval_set=[(X_train, Y_train), (X_val, Y_val)],
            eval_metric=["logloss", "accuracy", "balanced_accuracy"],
            patience=patience,
            batch_size=batch_size,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=True,
            max_epochs=max_epochs,
            from_unsupervised=unsupervised_model
        )
        
        train_time = time.time() - train_start
        total_time = time.time() - start_time
        
        print(f">>> Addestramento supervisionato completato in {train_time:.2f}s ({train_time/60:.2f}min)")
        
        # Estrazione metriche
        val_loss = min(model.history["val_1_logloss"])
        val_acc = max(model.history["val_1_accuracy"])
        
        # Calcolo balanced accuracy
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(Y_val, y_pred)

        print("\n" + "-"*70)
        print(f"RISULTATI PER L'ESECUZIONE {i+1}/{n_comb}")
        print("-"*70)
        print(f"Tempo Totale:         {total_time:.2f}s ({total_time/60:.2f}min)")
        print(f"  - Pre-addestramento: {pretrain_time:.2f}s")
        print(f"  - Addestramento:     {train_time:.2f}s")
        print(f"Val Loss:             {val_loss:.6f}")
        print(f"Val Accuracy:         {val_acc:.4f}")
        print(f"Balanced Accuracy:    {balanced_acc:.4f}")
        print("-"*70)

        is_best = False
        if balanced_acc > best_balanced_acc:
            is_best = True
            best_loss = val_loss
            best_balanced_acc = balanced_acc
            best_model = copy.deepcopy(model)
            best_params = params
            best_time = total_time

            best_model.save_model(save_path)
            torch.save({
                "training_time": best_time,
                "val_loss": val_loss,
                "balanced_accuracy": best_balanced_acc,
                "hyperparameters": best_params
            }, save_path + "_meta.pth")

            print(f"\n✓ TROVATO NUOVO MIGLIOR MODELLO!")
            print(f"   Miglior Balanced Accuracy precedente: {best_balanced_acc:.4f} → Nuova migliore: {balanced_acc:.4f}")
            print(f"   Val Loss: {val_loss:.6f}")
            print(f"   Salvato in: {save_path}.zip")
        else:
            print(f"\n✗ Non migliore dell'attuale (Balanced Acc: {best_balanced_acc:.4f})")

        # Riassunto a fine iterazione
        print("\n" + "="*70)
        print(f"[ESECUZIONE {i+1}/{n_comb}] COMPLETATA")
        print(f"Stato: {'✓ NUOVO MIGLIORE' if is_best else '✗ Nessun miglioramento'}")
        print(f"Miglior Balanced Acc attuale: {best_balanced_acc:.4f}")
        print(f"Miglior Val Loss attuale: {best_loss:.6f}")
        remaining = n_comb - (i + 1)
        est_time_remaining = remaining * total_time
        print(f"Esecuzioni rimanenti: {remaining}")
        print(f"Tempo stimato rimanente: {est_time_remaining/60:.1f} minuti ({est_time_remaining/3600:.1f} ore)")
        print("="*70 + "\n")

    print("\n" + "="*70)
    print("GRID SEARCH TABNET COMPLETATA")
    print("="*70)
    print(f"Miglior Balanced Accuracy: {best_balanced_acc:.4f}")
    print(f"Miglior Val Loss:          {best_loss:.6f}")
    print(f"Miglior Configurazione:    {best_params}")
    print(f"Miglior Tempo Addestramento: {best_time:.2f}s ({best_time/60:.2f}min)")
    print(f"Modello salvato in:        {save_path}.zip")
    print("="*70)

    return {
        "best_model": best_model,
        "best_params": best_params,
        "best_loss": best_loss,
        "best_balanced_acc": best_balanced_acc,
        "best_time": best_time,
    }

In [ ]:
structure_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns=loan_performance_data_leakage + settlement_data_leakage + hardship_data_leakage + other_leakage + loan_contract_interest_rate)),
    ('drop_non_significant', ColumnDropper(columns=other_non_significant)),
    ('drop_high_nan', HighNanDropper(threshold=0.9)),
    ('drop_joint', ColumnDropper(columns=joint_and_secondary_cols)),
    ('high_correlation', HighlyCorrelatedDropper(threshold=0.95)),
    ('extract_nums', NumericExtractor(columns=number_from_string_cols)),
    ('avg_features', FeatureAverager(columns=average_cols, new_name='fico_average')),
    ('date_diff', DateDifferenceTransformer(reference_col=date_diff_reference, target_cols=date_diff_target)),
    ('rounding', RoundToIntTransformer(columns=round_to_nearest_int)),
])

complete_pipeline = CompletePipelineTabNet(
    structure_pipeline=structure_pipeline,
    random_state=SEED
)

train_val_data = create_train_val_split(X, y, complete_pipeline)

X_train_np = train_val_data['X_train_np']
X_val_np = train_val_data['X_val_np']
Y_train_np = train_val_data['Y_train_np']
Y_val_np = train_val_data['Y_val_np']
cat_idxs = train_val_data['cat_idxs']
cat_dims = train_val_data['cat_dims']


# Grid ampliata per la nuova macchina (GPU, ~3h). n_a e' legato a n_d dentro
# run_tabnet_grid_search (n_a = n_d, come gia' fa il pretrainer), quindi qui si varia solo n_d.
# batch_size 4096 -> meno step/epoca su GPU. 2(lr)*2(n_d)*2(n_steps)*2(gamma) = 16 combinazioni.
param_grid = {
    'batch_size':    [4096],
    'epochs':        [60],
    'epsilon':       [1e-15],
    'lr':            [0.01, 0.02],
    'n_d':           [32, 64],
    'n_steps':       [3, 5],
    'gamma':         [1.3, 1.5],
    'n_independent': [2],
    'n_shared':      [2],
    'patience':      [15]
}

results = run_tabnet_grid_search(
    param_grid=param_grid,
    X_train=X_train_np,
    Y_train=Y_train_np,
    X_val=X_val_np,
    Y_val=Y_val_np,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    save_path="tabnet_best_model"
)

best_model = results['best_model']
y_pred = best_model.predict(X_val_np)

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(Y_val_np, y_pred))

cm = confusion_matrix(Y_val_np, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - TabNet')
plt.show()

# Save the Structural Pipeline
with open("tabnet_structural_pipeline.pkl", "wb") as f:
    pickle.dump(structure_pipeline, f)
print("Pipeline saved.")